# Topology, Density, and C--C Topology Results

Loads CREAM results for full-DAG topology, full-DAG density, and C--C fixed-count topology experiments. Each section includes original CREAM as a reference row/curve when available.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_colwidth', 180)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'experiments').exists() and (PROJECT_ROOT.parent / 'experiments').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASETS = {
    'celeba': {
        'display': 'CelebA',
        'model': 'Standard_CelebA',
        'original_experiment': 'CREAM_best_celeba',
    },
    'cub': {
        'display': 'CUB',
        'model': 'Standard_CUB',
        'original_experiment': 'CREAM_best',
    },
    'cfmnist': {
        'display': 'Complete_Concept_FMNIST',
        'model': 'Standard_FashionMNIST',
        'original_experiment': 'CREAM_best_cfmnist',
    },
}

FAMILIES = {
    'topology_random_full': 'Full-DAG topology random, fixed full-DAG edge count',
    'density_random_full': 'Full-DAG density random, varied full-DAG edge count',
    'concept_concept_fixcount_topology': 'C--C-only topology replacement, fixed C--C edge count',
}

KEY_METRICS = [
    'dataset', 'family', 'variant', 'is_original_reference',
    'test_task_accuracy', 'test_concept_accuracy', 'CCI',
    'PFI_concept_importance', 'PFI_side_importance', 'concept_leakage',
    'c2y_baseline_accuracy', 'test_dropout_task_accuracy',
    'metrics_path',
]


def experiment_root(dataset, family):
    meta = DATASETS[dataset]
    return (
        PROJECT_ROOT / 'experiments' / meta['display'] / 'train_cbm' / meta['model'] /
        f'{dataset}_{family}' / f'CREAM_{dataset}_{family}'
    )


def original_root(dataset):
    meta = DATASETS[dataset]
    return (
        PROJECT_ROOT / 'experiments' / meta['display'] / 'train_cbm' / meta['model'] /
        'CREAM' / meta['original_experiment']
    )


def read_metric_row(csv_path):
    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        print(f'Skipping unreadable metrics CSV {csv_path}: {exc}')
        return None
    if df.empty:
        return None
    return df.iloc[0].to_dict()


def choose_metric_csv(root):
    candidates = sorted((root / 'last_metrics').glob('*.csv'))
    candidates += sorted(root.glob('**/last_metrics/*.csv'))
    # Do not scan arbitrary root CSVs here; intermediate activation CSVs can be large.
    seen = set()
    deduped = []
    for path in candidates:
        key = str(path.resolve()) if path.exists() else str(path)
        if key not in seen:
            deduped.append(path)
            seen.add(key)
    return deduped[-1] if deduped else None


def load_result_metrics():
    rows = []
    for dataset in DATASETS:
        # Original CREAM reference once per family, so every section can display it.
        o_root = original_root(dataset)
        o_csv = choose_metric_csv(o_root)
        if o_csv is not None:
            row = read_metric_row(o_csv)
            if row is not None:
                for family in FAMILIES:
                    copied = dict(row)
                    copied.update({
                        'dataset': dataset,
                        'family': family,
                        'variant': 'original_CREAM',
                        'is_original_reference': True,
                        'metrics_path': str(o_csv),
                    })
                    rows.append(copied)
        else:
            print(f'Original CREAM metrics not found for {dataset}: {o_root}')

        for family in FAMILIES:
            root = experiment_root(dataset, family)
            if not root.exists():
                print(f'Result root not found yet for {dataset}/{family}: {root}')
                continue
            for variant_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
                csv_path = choose_metric_csv(variant_dir)
                if csv_path is None:
                    continue
                row = read_metric_row(csv_path)
                if row is None:
                    continue
                row.update({
                    'dataset': dataset,
                    'family': family,
                    'variant': variant_dir.name,
                    'is_original_reference': False,
                    'metrics_path': str(csv_path),
                })
                rows.append(row)
    return pd.DataFrame(rows)


def intervention_accuracy_column(df):
    for col in ['test_task_accuracy', 'task_accuracy', 'accuracy', 'test_accuracy', 'test_acc']:
        if col in df.columns:
            return col
    matches = [col for col in df.columns if 'accuracy' in col.lower() or col.lower().endswith('_acc')]
    return matches[0] if matches else None


def choose_intervention_csv(root):
    candidates = sorted(root.glob('lightning_logs/**/intervention_results.csv'))
    candidates += sorted(root.glob('**/intervention_results.csv'))
    seen = set()
    deduped = []
    for path in candidates:
        key = str(path.resolve()) if path.exists() else str(path)
        if key not in seen:
            deduped.append(path)
            seen.add(key)
    return deduped[-1] if deduped else None


def load_result_interventions():
    rows = []
    for dataset in DATASETS:
        o_root = original_root(dataset)
        o_csv = choose_intervention_csv(o_root)
        if o_csv is not None:
            df = pd.read_csv(o_csv)
            if not df.empty:
                for family in FAMILIES:
                    copied = df.copy()
                    copied['dataset'] = dataset
                    copied['family'] = family
                    copied['variant'] = 'original_CREAM'
                    copied['is_original_reference'] = True
                    copied['intervention_path'] = str(o_csv)
                    rows.append(copied)
        else:
            print(f'Original CREAM intervention curve not found for {dataset}: {o_root}')

        for family in FAMILIES:
            root = experiment_root(dataset, family)
            if not root.exists():
                continue
            for variant_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
                csv_path = choose_intervention_csv(variant_dir)
                if csv_path is None:
                    continue
                df = pd.read_csv(csv_path)
                if df.empty:
                    continue
                df['dataset'] = dataset
                df['family'] = family
                df['variant'] = variant_dir.name
                df['is_original_reference'] = False
                df['intervention_path'] = str(csv_path)
                rows.append(df)
    return pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()


def available(columns, requested):
    return [col for col in requested if col in columns]


def display_family_results(family, metrics_df, interventions_df):
    print(f"\n=== {family}: {FAMILIES[family]} ===")
    fam_metrics = metrics_df[metrics_df['family'] == family].copy() if not metrics_df.empty else pd.DataFrame()
    if fam_metrics.empty:
        print('No metrics found yet for this family.')
    else:
        cols = available(fam_metrics.columns, KEY_METRICS)
        sort_cols = available(fam_metrics.columns, ['dataset', 'is_original_reference', 'variant'])
        table = fam_metrics[cols].sort_values(sort_cols, ascending=[True, False, True] if len(sort_cols) == 3 else True)
        display(table)

    fam_interventions = interventions_df[interventions_df['family'] == family].copy() if not interventions_df.empty else pd.DataFrame()
    if fam_interventions.empty:
        print('No intervention curves found yet for this family.')
        return
    acc_col = intervention_accuracy_column(fam_interventions)
    if acc_col is None or 'num_interventions' not in fam_interventions.columns:
        print('Intervention CSVs found, but no plottable accuracy/num_interventions columns.')
        display(fam_interventions.head())
        return

    for dataset, dataset_df in fam_interventions.groupby('dataset'):
        group_values = [None]
        if 'group_interventions' in dataset_df.columns:
            group_values = sorted(dataset_df['group_interventions'].dropna().unique())
        for group_value in group_values:
            df = dataset_df if group_value is None else dataset_df[dataset_df['group_interventions'] == group_value]
            if df.empty:
                continue
            fig, ax = plt.subplots(figsize=(8.5, 5))
            for variant, variant_df in df.groupby('variant'):
                variant_df = variant_df.dropna(subset=['num_interventions', acc_col]).sort_values('num_interventions')
                if variant_df.empty:
                    continue
                is_original = bool(variant_df['is_original_reference'].iloc[0]) if 'is_original_reference' in variant_df.columns else False
                linewidth = 3.0 if is_original else 1.8
                linestyle = '--' if is_original else '-'
                ax.plot(
                    variant_df['num_interventions'],
                    variant_df[acc_col],
                    marker='o',
                    linewidth=linewidth,
                    linestyle=linestyle,
                    label=variant,
                )
            suffix = '' if group_value is None else f' | group_interventions={group_value}'
            ax.set_title(f'{dataset}: {family} intervention curve{suffix}')
            ax.set_xlabel('Number of interventions')
            ax.set_ylabel(acc_col)
            ax.grid(True, alpha=0.3)
            ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
            plt.tight_layout()
            plt.show()


## Load Results


In [ ]:
metrics = load_result_metrics()
interventions = load_result_interventions()

print(f'Loaded metric rows: {len(metrics)}')
print(f'Loaded intervention rows: {len(interventions)}')

if not metrics.empty:
    metrics_out = PROJECT_ROOT / 'notebook/topology_density_c2c_results_metrics.csv'
    metrics.to_csv(metrics_out, index=False)
    print(f'Saved metrics table to {metrics_out}')

if not interventions.empty:
    interventions_out = PROJECT_ROOT / 'notebook/topology_density_c2c_results_interventions.csv'
    interventions.to_csv(interventions_out, index=False)
    print(f'Saved intervention table to {interventions_out}')


## All Result Metrics


In [ ]:
if metrics.empty:
    print('No metrics found yet. Run the Condor jobs first, then rerun this notebook.')
else:
    cols = available(metrics.columns, KEY_METRICS)
    display(metrics[cols].sort_values(['family', 'dataset', 'is_original_reference', 'variant'], ascending=[True, True, False, True]))


## Full-DAG Topology

Family key: `topology_random_full`


In [ ]:
display_family_results('topology_random_full', metrics, interventions)


## Full-DAG Density

Family key: `density_random_full`


In [ ]:
display_family_results('density_random_full', metrics, interventions)


## C--C Fixed-Count Topology

Family key: `concept_concept_fixcount_topology`


In [ ]:
display_family_results('concept_concept_fixcount_topology', metrics, interventions)
